# Vehicle Cabin Temperature Predictor — Exploratory Analysis

This notebook loads the training dataset (14 vehicles, 19 time steps each) and walks through:
1. Data overview
2. All 14 vehicle cooling curves on one chart
3. Fitted two-phase Newton curve vs. actual data for 3 representative vehicles
4. Feature correlation heatmap
5. Feature importance (Ridge and Random Forest)


In [ ]:
import sys
sys.path.insert(0, '..')  # make backend importable from notebooks/

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Training Data

The dataset contains 14 vehicles. Each row represents one vehicle with:
- **14 raw input features** (heat load, cabin volume, compressor size, EBHS, etc.)
- **19 temperature readings** sampled at 5-minute intervals from t=0 to t=90 min

T_0min is the *soak temperature* — the cabin temperature before the AC is switched on.

In [ ]:
DATA_PATH = '../backend/data/vehicles_combined.csv'
TIME_POINTS = list(range(0, 95, 5))  # 0, 5, ..., 90
TEMP_COLS = [f'T_{t}min' for t in TIME_POINTS]

FEATURE_COLS = [
    'heat_load_kw', 'cabin_volume_m3', 'pulley_ratio', 'solar_w_m2',
    'ac_unit_capacity_kw', 'condenser_capacity_kw', 'compressor_size_cc',
    'airflow_m3_hr', 'soaking_time_hr', 'rpm_0_30', 'rpm_31_50',
    'rpm_51_70', 'rpm_71_90', 'ebhs',
]

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Vehicles: {df["vehicle"].tolist()}')
df[FEATURE_COLS].describe()

In [ ]:
# Temperature range across all vehicles
temps_all = df[TEMP_COLS].values
print(f'Soak temperature range: {temps_all[:, 0].min():.1f} – {temps_all[:, 0].max():.1f} °C')
print(f'Final temperature range (T_90min): {temps_all[:, -1].min():.1f} – {temps_all[:, -1].max():.1f} °C')
print(f'Max pull-down: {(temps_all[:, 0] - temps_all[:, -1]).max():.1f} °C')

## 2. All 14 Vehicle Cooling Curves

Each line is one vehicle. The rapid initial drop (minutes 0–20) is consistent across all vehicles — cabin cooling is essentially complete well before the 30-minute mark for every vehicle in the training set.

In [ ]:
fig = go.Figure()

colors = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24

for i, (_, row) in enumerate(df.iterrows()):
    temps = [float(row[c]) for c in TEMP_COLS]
    fig.add_trace(go.Scatter(
        x=TIME_POINTS,
        y=temps,
        mode='lines+markers',
        name=row['vehicle'],
        line=dict(color=colors[i % len(colors)], width=2),
        marker=dict(size=5),
        hovertemplate='%{fullData.name}<br>t=%{x} min<br>T=%{y:.1f} °C<extra></extra>',
    ))

fig.update_layout(
    title='Cabin Cool-Down Curves — All 14 Training Vehicles',
    xaxis_title='Time (min)',
    yaxis_title='Cabin Temperature (°C)',
    legend_title='Vehicle',
    height=550,
    hovermode='x unified',
    template='plotly_white',
)
fig.add_vline(x=30, line_dash='dash', line_color='gray',
              annotation_text='Segment 1|2 boundary', annotation_position='top')
fig.show()

## 3. Fitted Two-Phase Newton Curve vs. Actual Data

The Physics-Ridge model fits a two-phase Newton cooling curve to each vehicle:

$$T(t) = T_{\text{final}} + (T_{\text{soak}} - T_{\text{final}}) \cdot e^{-t/\tau_1} \quad (t \leq 70)$$
$$T(t) = T_{\text{final}} + (T_{70} - T_{\text{final}}) \cdot e^{-(t-70)/\tau_2} \quad (t > 70)$$

Below we show the fit for three vehicles: V1 (typical), V5 (small cabin, low T_final), and V9 (large cabin, high EBHS).

In [ ]:
T_BREAK = 70  # phase boundary (minutes)

def fit_two_phase(time_arr, temps):
    T_soak = temps[0]

    def model(t, tau1, T_final, tau2):
        T_at_70 = T_final + (T_soak - T_final) * np.exp(-T_BREAK / tau1)
        return np.where(
            t <= T_BREAK,
            T_final + (T_soak - T_final) * np.exp(-t / tau1),
            T_final + (T_at_70 - T_final) * np.exp(-(t - T_BREAK) / tau2),
        )

    try:
        popt, _ = curve_fit(
            model, time_arr, temps,
            p0=[5.0, 28.0, 8.0],
            bounds=([0.5, 20.0, 0.5], [20.0, 70.0, 40.0]),
            maxfev=10000,
        )
        tau1, T_final, tau2 = popt
    except Exception:
        tau1, T_final, tau2 = 5.0, float(temps[-1]), 8.0

    T_at_70 = T_final + (temps[0] - T_final) * np.exp(-T_BREAK / tau1)
    fitted = np.where(
        time_arr <= T_BREAK,
        T_final + (temps[0] - T_final) * np.exp(-time_arr / tau1),
        T_final + (T_at_70 - T_final) * np.exp(-(time_arr - T_BREAK) / tau2),
    )
    return fitted, tau1, T_final, tau2


SHOWCASE_VEHICLES = ['V1', 'V5', 'V9']
time_arr = np.array(TIME_POINTS, dtype=float)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=SHOWCASE_VEHICLES,
    shared_yaxes=True,
)

for col, vname in enumerate(SHOWCASE_VEHICLES, start=1):
    row_data = df[df['vehicle'] == vname].iloc[0]
    actual = np.array([float(row_data[c]) for c in TEMP_COLS])
    fitted, tau1, T_final, tau2 = fit_two_phase(time_arr, actual)

    fig.add_trace(go.Scatter(
        x=TIME_POINTS, y=actual.tolist(),
        mode='markers', name=f'{vname} actual',
        marker=dict(size=7, color='steelblue'),
        showlegend=(col == 1),
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=TIME_POINTS, y=fitted.tolist(),
        mode='lines', name=f'{vname} fitted',
        line=dict(color='crimson', width=2),
        showlegend=(col == 1),
    ), row=1, col=col)

    fig.add_annotation(
        text=f'τ₁={tau1:.1f} min<br>T_final={T_final:.1f}°C',
        xref=f'x{col}', yref=f'y{col}',
        x=60, y=actual.max() - 3,
        showarrow=False,
        font=dict(size=11),
        bgcolor='rgba(255,255,255,0.7)',
        row=1, col=col,
    )

fig.update_layout(
    title='Two-Phase Newton Curve Fit vs. Actual Data',
    height=420,
    template='plotly_white',
)
fig.update_xaxes(title_text='Time (min)')
fig.update_yaxes(title_text='Temperature (°C)', col=1)
fig.show()

## 4. Feature Correlation Heatmap

Pearson correlation between all 14 raw input features. Strong correlations (positive or negative) indicate features that carry similar information — useful for spotting multicollinearity before feeding into Ridge regression.

In [ ]:
corr = df[FEATURE_COLS].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu',
    zmid=0,
    zmin=-1,
    zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
    hovertemplate='%{y} vs %{x}<br>r = %{z:.3f}<extra></extra>',
))

fig.update_layout(
    title='Feature Correlation Matrix (Pearson r) — Raw Input Features',
    height=600,
    width=750,
    template='plotly_white',
    xaxis=dict(tickangle=45),
)
fig.show()

# Highlight the strongest correlations
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['Feature A', 'Feature B', 'r']
print('Top 10 strongest correlations:')
print(corr_pairs.reindex(corr_pairs['r'].abs().sort_values(ascending=False).index).head(10).to_string(index=False))

## 5. Feature Importance

We compare feature importance across two models:
- **Ridge (T_final):** standardised absolute coefficients for the T_final model — shows which engineered features most strongly predict the steady-state equilibrium temperature
- **Random Forest:** Gini impurity-based feature importance — reflects which features most reduce prediction error across all 266 training samples

The models often agree on the top features but differ in ranking mid-tier ones, revealing linear vs. non-linear structure in the data.

In [ ]:
# Reproduce the engineered features needed
def engineer_df(df):
    d = df.copy()
    d['ac_power_phase1']        = d['compressor_size_cc'] * d['pulley_ratio'] * d['rpm_0_30']  / 1e6
    d['ac_power_phase2']        = d['compressor_size_cc'] * d['pulley_ratio'] * d['rpm_71_90'] / 1e6
    d['ac_power_0_30']          = d['compressor_size_cc'] * d['pulley_ratio'] * d['rpm_0_30']  / 1e6
    d['ac_power_31_50']         = d['compressor_size_cc'] * d['pulley_ratio'] * d['rpm_31_50'] / 1e6
    d['ac_power_51_70']         = d['compressor_size_cc'] * d['pulley_ratio'] * d['rpm_51_70'] / 1e6
    d['ac_power_71_90']         = d['compressor_size_cc'] * d['pulley_ratio'] * d['rpm_71_90'] / 1e6
    d['heat_density']           = d['heat_load_kw'] / d['cabin_volume_m3']
    d['cooling_effectiveness']  = d['airflow_m3_hr'] / d['cabin_volume_m3']
    d['rpm_drop']               = d['rpm_51_70'] - d['rpm_71_90']
    d['airflow_heat_ratio']     = (d['airflow_m3_hr'] * 1.2 * 1.006 * 10 / 3600) / d['heat_load_kw']
    d['solar_gain']             = d['solar_w_m2'] * d['cabin_volume_m3'] / 1000
    d['net_cooling_power']      = d['ac_unit_capacity_kw'] - d['heat_load_kw'] - d['ebhs'] * 0.003 - d['solar_w_m2'] * 0.001
    d['ebhs_heat_fraction']     = d['ebhs'] * 0.003
    d['heat_load_fraction']     = d['heat_load_kw'] / d['ac_unit_capacity_kw']
    d['ac_per_volume']          = d['ac_unit_capacity_kw'] / d['cabin_volume_m3']
    d['net_cooling_per_volume'] = d['net_cooling_power'] / d['cabin_volume_m3']
    d['heat_balance_ratio']     = (d['heat_load_kw'] + d['ebhs'] * 0.003 + d['solar_w_m2'] * 0.001) / d['ac_unit_capacity_kw']
    d['sealing_quality']        = 1.0 / (1.0 + d['ebhs'] / 100.0)
    d['infiltration_airflow_ratio'] = d['ebhs'] / d['airflow_m3_hr']
    d['thermal_mass']           = d['cabin_volume_m3'] * 1.2 * 1.006
    d['tau_physics']            = d['thermal_mass'] / d['net_cooling_power'].clip(lower=0.1)
    return d

df_eng = engineer_df(df)

# Fit T_final for each vehicle
T_finals = []
for _, row in df.iterrows():
    temps = row[TEMP_COLS].values.astype(float)
    _, tau1, T_final, tau2 = fit_two_phase(time_arr, temps)
    T_finals.append(T_final)
df_eng['T_final'] = T_finals
print('Engineered dataframe shape:', df_eng.shape)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

ENG_FEATS = [
    'ac_power_phase1', 'ac_power_phase2',
    'heat_density', 'cooling_effectiveness', 'rpm_drop',
    'airflow_heat_ratio', 'solar_gain',
    'net_cooling_power', 'ebhs_heat_fraction', 'heat_load_fraction',
    'ac_per_volume', 'net_cooling_per_volume', 'heat_balance_ratio',
    'sealing_quality', 'infiltration_airflow_ratio', 'thermal_mass', 'tau_physics',
    'ac_power_0_30', 'ac_power_31_50', 'ac_power_51_70', 'ac_power_71_90',
]
TFINAL_FEATS = ['net_cooling_power', 'sealing_quality', 'heat_load_fraction',
                'ac_per_volume', 'net_cooling_per_volume']
RF_FEATS = ENG_FEATS + ['time_min']

# Ridge importance for T_final
sc_tf = StandardScaler()
X_tf = sc_tf.fit_transform(df_eng[TFINAL_FEATS].values)
ridge_tf = Ridge(alpha=1.0).fit(X_tf, df_eng['T_final'].values)
abs_coefs = np.abs(ridge_tf.coef_)
ridge_imp = abs_coefs / abs_coefs.max()

# Random Forest
rows_rf = []
for _, row in df_eng.iterrows():
    base = [row[f] for f in ENG_FEATS]
    for t in TIME_POINTS:
        rows_rf.append(base + [float(t), float(row[f'T_{t}min'])])
arr_rf = np.array(rows_rf, dtype=float)
X_rf, y_rf = arr_rf[:, :-1], arr_rf[:, -1]
rf = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42)
rf.fit(X_rf, y_rf)
rf_imp = rf.feature_importances_
rf_imp_norm = rf_imp / rf_imp.max()

print('Models trained successfully.')

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Ridge — T_final Model (normalised |coef|)',
        'Random Forest — Feature Importance (normalised)',
    ],
    horizontal_spacing=0.12,
)

# Ridge (T_final)
order_ridge = np.argsort(ridge_imp)
fig.add_trace(go.Bar(
    x=ridge_imp[order_ridge],
    y=[TFINAL_FEATS[i] for i in order_ridge],
    orientation='h',
    marker_color='steelblue',
    showlegend=False,
), row=1, col=1)

# Random Forest
order_rf = np.argsort(rf_imp_norm)[-15:]  # top 15 features
fig.add_trace(go.Bar(
    x=rf_imp_norm[order_rf],
    y=[RF_FEATS[i] for i in order_rf],
    orientation='h',
    marker_color='coral',
    showlegend=False,
), row=1, col=2)

fig.update_layout(
    title='Feature Importance Comparison: Ridge vs. Random Forest',
    height=520,
    template='plotly_white',
)
fig.update_xaxes(title_text='Normalised Importance')
fig.show()

print('\nRidge T_final — ranked features:')
for f, imp in sorted(zip(TFINAL_FEATS, ridge_imp), key=lambda x: -x[1]):
    print(f'  {f:<30s}  {imp:.4f}')

print('\nRandom Forest — top 10 features:')
for f, imp in sorted(zip(RF_FEATS, rf_imp_norm), key=lambda x: -x[1])[:10]:
    print(f'  {f:<30s}  {imp:.4f}')

## Summary

| Finding | Detail |
|---|---|
| **Top Ridge feature** | `net_cooling_power` — integrates AC capacity, heat load, EBHS, and solar gain |
| **Top RF feature** | `airflow_heat_ratio` — non-linear interaction not captured by Ridge |
| **Cooling speed** | τ₁ = 3–5 min for all vehicles; cabin near steady state by t=20 min |
| **EBHS effect** | Appears in `sealing_quality`, `net_cooling_power`, and `heat_balance_ratio` — central driver of T_final |
| **Dataset size** | 14 vehicles is small; Ridge regularisation and EBHS physics grounding help prevent overfit |
